# Speeding up MatmulT (Graviton4)

Note: expect max 400 GiB/s.

## 2025-11-11

Comparing with ACL (note that measurements have persistent noise between runs)

```sh
./dev run benchmark -- _tensor_proj
$ build/release/benchmark _tensor_proj
-- Running benchmark: _tensor_proj
            _tensor_proj.text.generate.mlp.up: 3.47922 ms, 16.8774 GMAC/s, 31.4465 GiB/s
          _tensor_proj.text.generate.mlp.down: 4.61875 ms, 12.7135 GMAC/s, 23.6881 GiB/s
        _tensor_proj.text.generate.attn.[q,o]: 0.237939 ms, 70.5106 GMAC/s, 131.4 GiB/s
        _tensor_proj.text.generate.attn.[k,v]: 0.0620893 ms, 67.5528 GMAC/s, 125.981 GiB/s
           _tensor_proj.text.generate.predict: 2.17964 ms, 241.02 GMAC/s, 449.048 GiB/s
             _tensor_proj.text.prefill.mlp.up: 2.19437 ms, 3425.21 GMAC/s, 51.8461 GiB/s
           _tensor_proj.text.prefill.mlp.down: 2.54369 ms, 2954.83 GMAC/s, 44.7261 GiB/s
         _tensor_proj.text.prefill.attn.[q,o]: 0.773503 ms, 2776.31 GMAC/s, 42.9256 GiB/s
         _tensor_proj.text.prefill.attn.[k,v]: 0.167799 ms, 3199.48 GMAC/s, 53.8334 GiB/s
                   _tensor_proj.vision.mlp.up: 2.53084 ms, 4145.78 GMAC/s, 12.3644 GiB/s
                 _tensor_proj.vision.mlp.down: 2.6713 ms, 3927.8 GMAC/s, 11.7143 GiB/s
           _tensor_proj.vision.attn.[q,k,v,o]: 0.654625 ms, 4007 GMAC/s, 16.3237 GiB/s
```

Against ACL v52.6.0 (FP16 as BFLOAT16 doesn't seem to work for me):

```sh
$ clang++ -stdlib=libc++ -isystem acl -isystem acl/include -O3 -Wall -Werror -fopenmp bench_acl.cpp -o bin/bench_acl -L acl/lib/armv8a-neon/ -larm_compute && env LD_LIBRARY_PATH=acl/lib/armv8a-neon ./bin/bench_acl
[bench_acl] running benchmarks
     text.generate.mlp.up: 1.66615 ms, 35.2431 GMAC/s, 65.6659 GiB/s    (1, 4096, 14336)
   text.generate.mlp.down: 2.66096 ms, 22.0673 GMAC/s, 41.1165 GiB/s    (1, 14336, 4096)
 text.generate.attn.[q,o]: 0.774833 ms, 21.6527 GMAC/s, 40.351 GiB/s    (1, 4096, 4096)
 text.generate.attn.[k,v]: 0.076678 ms, 54.7003 GMAC/s, 102.012 GiB/s    (1, 4096, 1024)
    text.generate.predict: 8.66938 ms, 60.5968 GMAC/s, 112.899 GiB/s    (1, 4096, 128256)
      text.prefill.mlp.up: 3.23899 ms, 2320.54 GMAC/s, 35.1251 GiB/s    (128, 4096, 14336)
    text.prefill.mlp.down: 4.24833 ms, 1769.21 GMAC/s, 26.7799 GiB/s    (128, 14336, 4096)
  text.prefill.attn.[q,o]: 1.14532 ms, 1875 GMAC/s, 28.9902 GiB/s    (128, 4096, 4096)
  text.prefill.attn.[k,v]: 0.250157 ms, 2146.14 GMAC/s, 36.1101 GiB/s    (128, 4096, 1024)
            vision.mlp.up: 2.67677 ms, 3919.77 GMAC/s, 11.6904 GiB/s    (1601, 1280, 5120)
          vision.mlp.down: 2.25137 ms, 4660.42 GMAC/s, 13.8993 GiB/s    (1601, 5120, 1280)
    vision.attn.[q,k,v,o]: 0.615264 ms, 4263.34 GMAC/s, 17.368 GiB/s    (1601, 1280, 1280)
```

## 2025-11-05

Use `bfmmla` (2x4x2) in a parallel loop of (8x8x8), blocked (16x(k)x16) for sake of L2 cache.

```sh
$ ./dev run benchmark -- _tensor_proj
-- Running benchmark: _tensor_proj
            _tensor_proj.text.generate.mlp.up: 0.334422 ms, 175.587 GMAC/s, 327.159 GiB/s
          _tensor_proj.text.generate.mlp.down: 0.357974 ms, 164.035 GMAC/s, 305.635 GiB/s
        _tensor_proj.text.generate.attn.[q,o]: 0.137139 ms, 122.337 GMAC/s, 227.982 GiB/s
        _tensor_proj.text.generate.attn.[k,v]: 0.123852 ms, 33.8656 GMAC/s, 63.1566 GiB/s
           _tensor_proj.text.generate.predict: 2.29336 ms, 229.069 GMAC/s, 426.781 GiB/s
             _tensor_proj.text.prefill.mlp.up: 2.25643 ms, 3331.01 GMAC/s, 50.4202 GiB/s
           _tensor_proj.text.prefill.mlp.down: 3.06498 ms, 2452.28 GMAC/s, 37.1192 GiB/s
         _tensor_proj.text.prefill.attn.[q,o]: 0.742824 ms, 2890.97 GMAC/s, 44.6985 GiB/s
         _tensor_proj.text.prefill.attn.[k,v]: 0.202402 ms, 2652.5 GMAC/s, 44.6301 GiB/s
                   _tensor_proj.vision.mlp.up: 2.54289 ms, 4126.14 GMAC/s, 12.3059 GiB/s
                 _tensor_proj.vision.mlp.down: 2.68742 ms, 3904.23 GMAC/s, 11.644 GiB/s
           _tensor_proj.vision.attn.[q,k,v,o]: 0.675135 ms, 3885.26 GMAC/s, 15.8278 GiB/s
```

## 2025-10-29

Replace the bf16 dot product with a `bfdot` loop (and `vld1q_bf16`).

```sh
$ ./dev run benchmark -- _tensor_proj
-- Running benchmark: _tensor_proj
            _tensor_proj.text.generate.mlp.up: 0.358274 ms, 163.898 GMAC/s, 305.379 GiB/s
          _tensor_proj.text.generate.mlp.down: 0.320161 ms, 183.408 GMAC/s, 341.732 GiB/s
        _tensor_proj.text.generate.attn.[q,o]: 0.209156 ms, 80.214 GMAC/s, 149.483 GiB/s
        _tensor_proj.text.generate.attn.[k,v]: 0.0482092 ms, 87.0021 GMAC/s, 162.252 GiB/s
           _tensor_proj.text.generate.predict: 2.26985 ms, 231.441 GMAC/s, 431.202 GiB/s
             _tensor_proj.text.prefill.mlp.up: 9.83482 ms, 764.243 GMAC/s, 11.568 GiB/s
           _tensor_proj.text.prefill.mlp.down: 12.32 ms, 610.08 GMAC/s, 9.23453 GiB/s
         _tensor_proj.text.prefill.attn.[q,o]: 2.83871 ms, 756.499 GMAC/s, 11.6965 GiB/s
         _tensor_proj.text.prefill.attn.[k,v]: 0.805554 ms, 666.462 GMAC/s, 11.2137 GiB/s
                   _tensor_proj.vision.mlp.up: 19.7181 ms, 532.115 GMAC/s, 1.58699 GiB/s
                 _tensor_proj.vision.mlp.down: 23.9557 ms, 437.988 GMAC/s, 1.30626 GiB/s
           _tensor_proj.vision.attn.[q,k,v,o]: 5.53047 ms, 474.296 GMAC/s, 1.93219 GiB/s
```

## 2025-10-17

```sh
$ ./dev run benchmark -- _tensor_proj
-- Running benchmark: _tensor_proj
            _tensor_proj.text.generate.mlp.up: 0.459857 ms, 127.692 GMAC/s, 237.92 GiB/s
          _tensor_proj.text.generate.mlp.down: 0.476373 ms, 123.265 GMAC/s, 229.672 GiB/s
        _tensor_proj.text.generate.attn.[q,o]: 0.139103 ms, 120.61 GMAC/s, 224.763 GiB/s
        _tensor_proj.text.generate.attn.[k,v]: 0.108685 ms, 38.5915 GMAC/s, 71.97 GiB/s
           _tensor_proj.text.generate.predict: 3.48178 ms, 150.882 GMAC/s, 281.11 GiB/s
             _tensor_proj.text.prefill.mlp.up: 49.3215 ms, 152.392 GMAC/s, 2.30669 GiB/s
           _tensor_proj.text.prefill.mlp.down: 49.8245 ms, 150.853 GMAC/s, 2.2834 GiB/s
         _tensor_proj.text.prefill.attn.[q,o]: 14.1076 ms, 152.221 GMAC/s, 2.35356 GiB/s
         _tensor_proj.text.prefill.attn.[k,v]: 3.65753 ms, 146.785 GMAC/s, 2.46975 GiB/s
                   _tensor_proj.vision.mlp.up: 68.5206 ms, 153.126 GMAC/s, 0.456687 GiB/s
                 _tensor_proj.vision.mlp.down: 73.0514 ms, 143.629 GMAC/s, 0.428362 GiB/s
           _tensor_proj.vision.attn.[q,k,v,o]: 17.8926 ms, 146.601 GMAC/s, 0.597225 GiB/s
```

# Attention

## 2025-10-29

```sh
$ ./dev run benchmark -- _tensor_attention
-- Running benchmark: _tensor_attention
         _tensor_attention.text.generate.self: 2.98542 ms, 0.175616 GMAC/s, 0.168666 GiB/s
        _tensor_attention.text.generate.cross: 6.9598 ms, 1.88445 GMAC/s, 0.879708 GiB/s
          _tensor_attention.text.prefill.self: 7.00742 ms, 9.57684 GMAC/s, 0.348403 GiB/s
         _tensor_attention.text.prefill.cross: 2.56072 ms, 655.585 GMAC/s, 3.14773 GiB/s
                     _tensor_attention.vision: 10.3869 ms, 631.737 GMAC/s, 1.46996 GiB/s
```

## 2025-10-17

```sh
$ ./dev run benchmark -- _tensor_attention
-- Running benchmark: _tensor_attention
         _tensor_attention.text.generate.self: 0.0426054 ms, 12.3057 GMAC/s, 11.8187 GiB/s
        _tensor_attention.text.generate.cross: 1.1012 ms, 11.9101 GMAC/s, 5.55993 GiB/s
          _tensor_attention.text.prefill.self: 1.36706 ms, 49.0899 GMAC/s, 1.78588 GiB/s
         _tensor_attention.text.prefill.cross: 17.6715 ms, 94.9985 GMAC/s, 0.456126 GiB/s
                     _tensor_attention.vision: 68.2231 ms, 96.1814 GMAC/s, 0.2238 GiB/s
```